In [1]:
import streamlit as st
import pandas as pd
from typing import List, Dict, Optional, Any
import plotly.express as px


In [2]:
bundesländer_data = pd.read_csv('../data/processed_data/test_digiclass05.csv')

In [3]:
bundesländer_data.columns

Index(['ISCO-Code', 'Anzahl', 'Baden-Württemberg', 'Bayern', 'Berlin',
       'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern',
       'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland',
       'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen',
       'Bezeichnung', 'is_supervisor', 'n_employees', 'self_employed',
       'control_work', 'control_daily', 'mod_stellung_im_beruf'],
      dtype='object')

In [4]:
path_livingstone =  "../data/processed_data/isco_livingstone.csv"
livingstone_data = pd.read_csv(path_livingstone, index_col=0)  # Fixed: now loads livingstone data

# Merge the datasets
bundesländer_data = pd.merge(bundesländer_data, livingstone_data, left_on='ISCO-Code', right_on='ISCO-Code')
#merged_data.drop(columns=['ISCO.Code'], inplace=True)

In [5]:

# Change 'Selbstständige ohne Beschäftigte' to 'Selbstständige'
bundesländer_data.loc[bundesländer_data['mod_stellung_im_beruf'] == 'Selbstständige ohne Beschäftigte', 'modifiziert_livingstone'] = 'Selbstständige'

# Keep 'Selbstständige mit Beschäftigten' as is 
# (this line isn't necessary if you don't need to change it, but included for clarity)
bundesländer_data.loc[bundesländer_data['mod_stellung_im_beruf'] == 'Selbstständige mit Beschäftigten', 'modifiziert_livingstone'] = 'Selbstständige mit Beschäftigten'


In [6]:
bundesländer_data

,ISCO-Code,Anzahl,Baden-Württemberg,Bayern,Berlin,Brandenburg,Bremen,Hamburg,Hessen,Mecklenburg-Vorpommern,...,Bezeichnung,is_supervisor,n_employees,self_employed,control_work,control_daily,mod_stellung_im_beruf,major_group,Berufsgattung(ISCO-Stufe 4),modifiziert_livingstone
0,1213,12900,2110,1950,610,320,0,0,980,0,...,NaN,1,0,0,4,2,Arbeiter*innen & Angestellte,Führungskräfte,Führungskräfte in Unternehmenspolitik und -pla...,Mittleres Management
1,1219,97710,14060,19300,4990,3290,500,2720,7470,920,...,NaN,1,0,0,4,2,Arbeiter*innen & Angestellte,Führungskräfte,Führungskräfte in der betrieblichen Verwaltung...,Mittleres Management
2,1221,162370,23170,29320,6840,4430,1030,4260,13410,1790,...,NaN,1,0,0,4,2,Arbeiter*innen & Angestellte,Führungskräfte,Führungskräfte in Vertrieb und Marketing,Mittleres Management
3,1222,29830,3540,4130,4860,470,220,2800,2610,0,...,NaN,1,0,0,4,2,Arbeiter*innen & Angestellte,Führungskräfte,Führungskräfte in Werbung und Öffentlichkeitsa...,Mittleres Management
4,1223,13740,3710,1980,800,300,0,0,1020,0,...,NaN,1,0,0,4,2,Arbeiter*innen & Angestellte,Führungskräfte,Führungskräfte in Forschung und Entwicklung,Mittleres Management
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1249,9622,0,0,0,0,0,0,0,0,0,...,Gelegenheitsarbeiter,0,0,1,4,2,Selbstständige ohne Beschäftigte,Hilfsarbeitskräfte,Gelegenheitsarbeiter,Selbstständige
1250,9623,1240,300,0,0,0,0,0,0,0,...,"Zählerableser, Automatenbefüller und -kassierer",1,5,1,4,2,Selbstständige mit Beschäftigten,Hilfsarbeitskräfte,"Zählerableser, Automatenbefüller und -kassierer",Selbstständige mit Beschäftigten
1251,9623,760,0,0,0,0,0,0,0,0,...,"Zählerableser, Automatenbefüller und -kassierer",0,0,1,4,2,Selbstständige ohne Beschäftigte,Hilfsarbeitskräfte,"Zählerableser, Automatenbefüller und -kassierer",Selbstständige
1252,9629,710,0,0,0,0,0,0,0,0,...,"Hilfsarbeitskräfte, anderweitig nicht genannt",1,5,1,4,2,Selbstständige mit Beschäftigten,Hilfsarbeitskräfte,"Hilfsarbeitskräfte, anderweitig nicht genannt",Selbstständige mit Beschäftigten


### Calculating percantages

In [7]:
# Define the columns_bundesländer to plot (Bundesländer)
columns_bundesländer = [
    'Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg',
    'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen',
    'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt',
    'Schleswig-Holstein', 'Thüringen'
]

result_df = bundesländer_data.copy()

# Calculate total for this Bundesland to get percentages
column_totals = bundesländer_data[columns_bundesländer].sum()

# Create new column names
pct_columns_bundesländer = [f"{col}_pct" for col in columns_bundesländer]
    
# Calculate all percentages at once using vectorized operations
# Handle division by zero by replacing NaN with 0
result_df[pct_columns_bundesländer] = (bundesländer_data[columns_bundesländer].div(column_totals) * 100).fillna(0)

In [8]:
# For show biggest Berufsgruppe per Bundesland
for bundesland in pct_columns_bundesländer:
    entity_with_highest = result_df.loc[result_df[bundesland].idxmax(), ['ISCO-Code', 'Berufsgattung(ISCO-Stufe 4)']]
    print(f"Entity with highest percentage in {bundesland}: {entity_with_highest['Berufsgattung(ISCO-Stufe 4)']} ({entity_with_highest['ISCO-Code']})")

Entity with highest percentage in Baden-Württemberg_pct: Verkäufer und Verkaufshilfskräfte in Handelsgeschäften (5223)
Entity with highest percentage in Bayern_pct: Verkäufer und Verkaufshilfskräfte in Handelsgeschäften (5223)
Entity with highest percentage in Berlin_pct: Verkäufer und Verkaufshilfskräfte in Handelsgeschäften (5223)
Entity with highest percentage in Brandenburg_pct: Verkäufer und Verkaufshilfskräfte in Handelsgeschäften (5223)
Entity with highest percentage in Bremen_pct: Verkäufer und Verkaufshilfskräfte in Handelsgeschäften (5223)
Entity with highest percentage in Hamburg_pct: Verkäufer und Verkaufshilfskräfte in Handelsgeschäften (5223)
Entity with highest percentage in Hessen_pct: Sekretariatskräfte (allgemein) (4120)
Entity with highest percentage in Mecklenburg-Vorpommern_pct: Verkäufer und Verkaufshilfskräfte in Handelsgeschäften (5223)
Entity with highest percentage in Niedersachsen_pct: Verkäufer und Verkaufshilfskräfte in Handelsgeschäften (5223)
Entity with 

In [9]:
result_df = result_df.groupby(['modifiziert_livingstone'])[pct_columns_bundesländer].sum().reset_index()

In [10]:
result_df

,modifiziert_livingstone,Baden-Württemberg_pct,Bayern_pct,Berlin_pct,Brandenburg_pct,Bremen_pct,Hamburg_pct,Hessen_pct,Mecklenburg-Vorpommern_pct,Niedersachsen_pct,Nordrhein-Westfalen_pct,Rheinland-Pfalz_pct,Saarland_pct,Sachsen_pct,Sachsen-Anhalt_pct,Schleswig-Holstein_pct,Thüringen_pct
0,Anleitende Beschäftigte,3.102722,3.195082,2.584520,2.465158,2.447540,2.880892,2.872989,2.490888,2.507176,2.405736,2.755733,2.398601,2.606930,2.294306,2.612770,2.460435
1,Dienstleistungsarbeiter*innen,25.954895,26.779192,25.537001,28.732167,31.694586,25.464014,28.756557,31.791285,29.214333,28.078629,29.231271,30.825175,26.805133,29.170312,30.690548,26.301600
2,Hochspezialisierte Beschäftigte,37.811905,36.141782,45.505947,35.906764,37.848704,46.082031,38.686614,33.078047,34.918984,37.592543,35.114554,36.876457,36.844681,32.538319,35.010693,34.439715
3,Industriearbeiter*innen,22.489337,22.069956,11.601336,22.537224,21.565861,13.073619,18.470710,24.081042,23.060663,20.993318,22.914457,23.531469,23.847725,28.295365,20.304893,28.304550
4,Mittleres Management,2.760394,2.695480,3.035683,2.745801,2.295891,3.285930,2.900669,2.108696,2.390956,2.629101,2.439510,1.888112,2.721131,2.250019,2.693671,2.534736
5,Selbstständige,3.426882,3.405868,7.365676,3.189287,1.594075,4.830418,3.857319,2.120501,3.047195,3.595638,2.960768,1.090909,3.206062,1.785541,2.940597,2.244961
6,Selbstständige mit Beschäftigten,4.265883,5.400515,4.165934,3.922078,2.186563,4.173899,4.210409,3.301017,4.292333,4.375116,4.015526,2.981352,3.657240,3.088240,4.913189,3.220431


In [12]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

category_colors = {
    'Dienstleistungsarbeiter*innen': '#0A1F44',
    'Hochspezialisierte Beschäftigte': '#F5B461',
    'Industriearbeiter*innen': '#FF6B6B'
}

rows, cols = 4, 4

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=columns_bundesländer,
    shared_yaxes=True,
    shared_xaxes=False
)

# One bar trace per state column
for idx, col_name in enumerate(pct_columns_bundesländer):
    row = idx // cols + 1
    col = idx % cols + 1

    fig.add_trace(
        go.Bar(
            x=result_df['modifiziert_livingstone'],
            y=result_df[col_name],
            #sets the color
            marker=dict(color=[category_colors[cat] for cat in result_df['modifiziert_livingstone']]),
            #makes xlabel disapear
            showlegend=(idx == 0),
            name=col_name
        ),
        row=row,
        col=col
    )

# Y-axis setup: range 0–100, ticks every 20
fig.update_yaxes(range=[0, 70], dtick=20)

# Hide x-axis tick labels
fig.update_xaxes(showticklabels=False)

# Clean layout
fig.update_layout(
    height=1000,
    width=1200,
    barmode='group',
    title_text="State Comparisons by Category",
    legend_title_text='Category'
)

fig.show()


KeyError: 'Anleitende Beschäftigte'

In [ ]:
digiclass = pd.read_csv('/Users/leonardhaas/code/streamlit/data/processed_data/added_digiclass.csv')
digiclass.head()

digiclass.dtypes

Unnamed: 0             int64
X                      int64
ISCO.Code              int64
Stellung.im.Beruf     object
is_supervisor          int64
self_employed          int64
n_employees          float64
Anzahl                 int64
control_work           int64
control_daily          int64
isco88                 int64
isco88com              int64
simple_wright         object
siops                float64
egp                   object
oesch                 object
dtype: object

In [ ]:
verdienst = pd.read_csv('/Users/leonardhaas/code/streamlit/data/processed_data/isco_verdienst.csv')
verdienst.head()

,Unnamed: 0,isco_08_key,fraktion,Berufsgattung(ISCO-Stufe 4),Anzahl,median_brutto_group_mean
0,0,110.0,Staatsangestellte,Offiziere in regulären Streitkräften,22030.0,5330.0
1,1,310.0,Staatsangestellte,Angehörige der regulären Streitkräfte in sonst...,107160.0,2627.0
2,2,1112.0,Top Management,Leitende Verwaltungsbedienstete,15660.0,8746.0
3,3,1114.0,Top Management,Leitende Bedienstete von Interessenorganisationen,6280.0,6307.0
4,4,1120.0,Top Management,Geschäftsführer und Vorstände,487470.0,8816.0


In [ ]:
#TODO semi duplicates filter

siops_verdienst_data = verdienst.merge(digiclass,left_on='isco_08_key',right_on='ISCO.Code')
siops_verdienst_data


,Unnamed: 0_x,isco_08_key,fraktion,Berufsgattung(ISCO-Stufe 4),Anzahl_x,median_brutto_group_mean,Unnamed: 0_y,X,ISCO.Code,Stellung.im.Beruf,...,n_employees,Anzahl_y,control_work,control_daily,isco88,isco88com,simple_wright,siops,egp,oesch
0,0,110.0,Staatsangestellte,Offiziere in regulären Streitkräften,22030.0,5330.0,434,433,110,Arbeiter*innen & Angestellte,...,NaN,21970,4,1,110,100,NaN,48.68,NaN,'Higher-grade managers and administrators'
1,0,110.0,Staatsangestellte,Offiziere in regulären Streitkräften,22030.0,5330.0,437,10,110,Selbstständige mit Beschäftigten,...,5.0,0,4,1,110,100,Self empl w/1-9 employees,48.68,NaN,NaN
2,0,110.0,Staatsangestellte,Offiziere in regulären Streitkräften,22030.0,5330.0,438,11,110,Selbstständige ohne Beschäftigte,...,0.0,0,4,1,110,100,Self empl w/no empoyees,48.68,NaN,NaN
3,1,310.0,Staatsangestellte,Angehörige der regulären Streitkräfte in sonst...,107160.0,2627.0,436,435,310,Arbeiter*innen & Angestellte,...,NaN,107060,4,1,110,100,NaN,43.23,NaN,NaN
4,1,310.0,Staatsangestellte,Angehörige der regulären Streitkräfte in sonst...,107160.0,2627.0,441,22,310,Selbstständige mit Beschäftigten,...,5.0,0,4,1,110,100,Self empl w/1-9 employees,43.23,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
988,329,9623.0,Dienstleistungsarbeiter,"Zählerableser, Automatenbefüller und -kassierer",9010.0,3625.0,1303,2608,9623,Selbstständige mit Beschäftigten,...,5.0,1240,4,1,9153,9153,Self empl w/1-9 employees,21.00,'IVb: Self-employed with no employees','Small business owners with employees'
989,329,9623.0,Dienstleistungsarbeiter,"Zählerableser, Automatenbefüller und -kassierer",9010.0,3625.0,1304,2609,9623,Selbstständige ohne Beschäftigte,...,0.0,760,4,1,9153,9153,Self empl w/no empoyees,21.00,'VIIa: Unskilled Worker','Small business owners without employees'
990,330,9629.0,Dienstleistungsarbeiter,"Hilfsarbeitskräfte, anderweitig nicht genannt",74020.0,2300.5,433,432,9629,Arbeiter*innen & Angestellte,...,NaN,72550,4,1,9100,9100,Low skilled workers,24.00,NaN,'Low-skilled manual'
991,330,9629.0,Dienstleistungsarbeiter,"Hilfsarbeitskräfte, anderweitig nicht genannt",74020.0,2300.5,1307,2620,9629,Selbstständige mit Beschäftigten,...,5.0,710,4,1,9100,9100,Self empl w/1-9 employees,24.00,'IIIa: Routine Nonmanual','Small business owners with employees'


In [ ]:
scatter_data=siops_verdienst_data[['oesch','Berufsgattung(ISCO-Stufe 4)','Anzahl_x','median_brutto_group_mean','siops']]#.drop_duplicates(inplace=True)

In [ ]:
working_class_only =siops_verdienst_data.loc[(siops_verdienst_data['Stellung.im.Beruf']=='Arbeiter*innen & Angestellte')&(siops_verdienst_data['is_supervisor']==0)]

In [ ]:
working_class_only = working_class_only[['siops','median_brutto_group_mean','oesch','Berufsgattung(ISCO-Stufe 4)','Anzahl_x','simple_wright']].drop_duplicates()

In [ ]:
working_class_only

Index(['siops', 'median_brutto_group_mean', 'oesch',
       'Berufsgattung(ISCO-Stufe 4)', 'Anzahl_x', 'simple_wright'],
      dtype='object')

In [ ]:
#TODO Sekrtariatsleiter sind drin why?
colors = [
    "#2f4f4f",  # darkslategray
    "#a0522d",  # sienna
    "#006400",  # darkgreen
    "#00008b",  # darkblue
    "#ff0000",  # red
    "#ffa500",  # orange
    "#ff69b4",   # hotpink
    "#00ff00",  # lime
    "#00fa9a",  # mediumspringgreen
    "#00ffff",  # aqua
    "#0000ff",  # blue
    "#d8bfd8",  # thistle
    "#ff00ff",  # fuchsia
    "#1e90ff",  # dodgerblue
    "#f0e68c",  # khaki
    "#ffff00",  # yellow
]



fig = px.scatter(
    working_class_only,
    x='siops',
    y='median_brutto_group_mean',
    size='Anzahl_x',  # Set the size of the markers
    #color='oesch',  # Set the color of the markers based on 'oesch'
    #symbol='oesch',
    hover_data=['Berufsgattung(ISCO-Stufe 4)', 'Anzahl_x'],
    title='Berufsgattungen nach Prestige und Verdienst',
    labels={'siops': 'Standard International Occupational Prestige-Scale(SIOPS)', 'median_brutto_group_mean': 'Monatlicher Brutto Verdienst'},
    #color_discrete_sequence=colors
)
fig.add_hline(y=4346, line_color="black", annotation_text="Median Bruttolohn 2024", annotation_position="top right")

#fig.add_hline(y=4323, line_dash="dash", line_color="grey", annotation_text="Durchschnitt Bruttolohn", annotation_position="top right")


fig.update_layout(
    width=900,
    height=800
)
# Show the plot
fig.show()

In [ ]:
import os 
def earnings_prestige_plot_data_set_up(file_name_verdienst,file_name):
    
    # current working directory
    notebook_dir = os.getcwd()
    file_path = os.path.join(notebook_dir, "data", "example.txt")
    path = os.getcwd()

In [ ]:
import os
from pathlib import Path

# Option 1: Use the current working directory
notebook_dir = os.getcwd()
file_path = os.path.join(notebook_dir, "example.txt")
file_path

'/Users/leonardhaas/code/streamlit/src/example.txt'

In [ ]:


def clean_group(df: pd.DataFrame,bundesländer_cols:list) -> pd.DataFrame:
   # Replace '/' with '0' and convert 'Insgesamt' column to integer
   df = df.replace('/', '0')
   df['Insgesamt'] = df['Insgesamt'].astype(int)
   
   # Calculate the total and percentage
   total = df['Insgesamt'].iloc[0]
   df['percent'] = (df['Insgesamt'] / total) * 100
   
   # Drop unnecessary columns and the first row
   df = df.drop(columns=bundesländer_cols)
   df = df.drop(index=0)
   
   return df

#@st.cache_data()
def read_data(file_path: str) -> (Dict[str, pd.DataFrame], List[str]):

    def load_sheets(file_path: str, sheet_names: List[str], header: int = 3) -> Dict[str, pd.DataFrame]:
        return {name: pd.read_excel(file_path, sheet_name=name, header=header) for name in sheet_names}
    
    sheet_names = pd.ExcelFile(file_path).sheet_names
    sheet_data = load_sheets(file_path, sheet_names[3:7])
    return sheet_data, sheet_names

def build_data_frames():
    file_path = '/Users/leonardhaas/code/streamlit/data/raw_data/Zensus22_Sonderauswertung_Haas.xlsx'

    sheet_data,sheet_names = read_data(file_path)   
    haupt_gruppen_1 = sheet_data[sheet_names[3]]
    berufs_gruppen_2 = sheet_data[sheet_names[4]]
    berufs_unter_gruppen_3 = sheet_data[sheet_names[5]]
    berufs_gattungen_4 = sheet_data[sheet_names[6]]


    bundesländer_cols =['Baden-Württemberg', 'Bayern',
        'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen',
        'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen',
        'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt',
        'Schleswig-Holstein', 'Thüringen']

    haupt_gruppen_1 = clean_group(haupt_gruppen_1, bundesländer_cols)
    berufs_gruppen_2 = clean_group(berufs_gruppen_2, bundesländer_cols)
    berufs_unter_gruppen_3 = clean_group(berufs_unter_gruppen_3, bundesländer_cols)
    berufs_gattungen_4 = clean_group(berufs_gattungen_4, bundesländer_cols)

    dataframes = {
        'Hauptgruppe (1-Str.)': haupt_gruppen_1,
        'Berufsgruppe (2-Str.)': berufs_gruppen_2,
        'Berufsuntergruppen (3-St.)': berufs_unter_gruppen_3,
        'Berufsgattung (4-St.)': berufs_gattungen_4
    }
    return dataframes

In [ ]:
data_frames = build_data_frames()

In [ ]:
data_level_4 = data_frames['Berufsgattung (4-St.)']
data_level_4['merg_id']=data_level_4['ISCO-Code'].map(lambda x: str(x)[:2])

In [ ]:
data_level_1 = data_frames['Hauptgruppe (1-Str.)']

In [ ]:
data_level_1.merge(data_level_4,left_on='ISCO-Code',right_on='merg_id')

,ISCO-Code_x,Bezeichnung_x,Insgesamt_x,percent_x,ISCO-Code_y,Bezeichnung_y,Insgesamt_y,percent_y,merg_id


In [ ]:
total=41043450
# Define the desired order of 'fraktion'
fraktion_order = [
    "Top Management","Mittleres Management","Anleitender Beschäftigter",
    "Klassisch selbständige Tätigkeit",
    "Staatsangestellte","Hochspezialisierte Beschäftigte", "Industriearbeiter",
    "Dienstleistungsarbeiter",

]

# Calculate the percentage and sort by the defined order
fraktion_percentage = (berufs_gattungen_4.groupby(['fraktion'])['Insgesamt'].sum() / total) * 100
fraktion_percentage = fraktion_percentage.reindex(fraktion_order)

# Display the result
print(fraktion_percentage)
#(berufs_gattungen_4.groupby(['fraktion'])['Insgesamt'].sum()/ total) * 100

In [ ]:

# Convert the fraktion_percentage series to a DataFrame for easier plotting
fraktion_percentage_df = fraktion_percentage.reset_index()

# Create the horizontal bar plot
fig = px.bar(
    fraktion_percentage_df,
    x='Insgesamt',
    y='fraktion',
    orientation='h',
    title='Klassenfraktionen nach Anteil an Erwerbstätigen Daten Zensus 2022',
    labels={'Insgesamt': 'Prozent(%)','fraktion':'Fraktion'},
    text='Insgesamt'  # Add text to show the count
)

# Update the layout to increase the left margin
fig.update_layout(
    margin=dict(l=220)  # Adjust the left margin as needed
)

# Update the text position and color
fig.update_traces(
    textposition='inside',  # Position the text outside the bar
    texttemplate='%{text:.2f}%',  # Format the text
    textfont=dict(color='white')  # Set the text color to white
)

# Show the plot
fig.show()

ISCO-Code       object
Bezeichnung     object
Insgesamt        int64
percent        float64
merg_id         object
dtype: object

## plot restaurantsbills like chart
https://plotly.com/python/horizontal-bar-charts/